# Dynamic Tool Loading & Calling

In [13]:
from langchain.tools import tool


from langchain_core.tools import tool


@tool
def standard_booking(movie_title: str) -> str:
    """Book a standard seat."""
    return f"Standard seat booked for {movie_title}."


@tool
def vip_lounge_booking(movie_title: str) -> str:
    """Book a VIP lounge seat with premium service. VIP members only."""
    return f"VIP lounge seat booked for {movie_title}."

In [14]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable


@wrap_model_call
def gate_vip_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Only expose vip_lounge_booking to VIP members."""

    print("I am here, this is my state:", request.state)

    is_vip = request.state.get("is_vip_member", False)

    if not is_vip:
        allowed = [
            t for t in request.tools
            if t.name != "vip_lounge_booking"
        ]

        request = request.override(tools=allowed)

    else:
        request = request.override(
            tools=request.tools
        )  # explicit no-op, VIP sees everything

    return handler(request)

In [25]:
from langchain.agents import create_agent, AgentState

class GatedState(AgentState):
    is_vip_member: bool

gated_agent_with_middleware = create_agent(
    model="openai:gpt-5-mini",
    tools=[standard_booking, vip_lounge_booking],
    middleware=[gate_vip_tools],
    state_schema=GatedState
)

In [16]:
def pretty_print_messages(result):
    for message in result["messages"]:
        message.pretty_print()

In [26]:
result_regular_user = gated_agent_with_middleware.invoke(
    {
        "messages": [
            ("user", "Book me a VIP lounge seat for Dune")
        ],
        "is_vip_member": False
    }
)

pretty_print_messages(result_regular_user)

# Expect: the tool genuinely isn't offered — the model can't book it
# because it never saw it.

I am here, this is my state: {'messages': [HumanMessage(content='Book me a VIP lounge seat for Dune', additional_kwargs={}, response_metadata={}, id='669e0619-06e3-42d6-aa92-c0d8abfaf772')], 'is_vip_member': False}
================================ Human Message =================================

Book me a VIP lounge seat for Dune
================================== Ai Message ==================================

I can help — I just need a few details before booking:

1. Which Dune movie do you mean (Dune 2021, Dune: Part Two, or something else)?  
2. City or specific theater, and preferred date and showtime (or time range).  
3. Number of seats.  
4. Full name and contact email/phone for the reservation.  
5. Do you specifically require a VIP lounge seat? (Note: my current booking tool only supports standard seats — if you want I can check VIP availability first and, if unavailable, either book a standard seat or wait for your instruction.)

Tell me those and I’ll proceed.


In [27]:
result_vip_user = gated_agent_with_middleware.invoke(
    {
        "messages": [
            ("user", "Book me a VIP lounge seat for Dune")
        ],
        "is_vip_member": True
    }
)

pretty_print_messages(result_vip_user)

I am here, this is my state: {'messages': [HumanMessage(content='Book me a VIP lounge seat for Dune', additional_kwargs={}, response_metadata={}, id='87793c03-3824-40df-be11-a9c243a43b61')], 'is_vip_member': True}
I am here, this is my state: {'messages': [HumanMessage(content='Book me a VIP lounge seat for Dune', additional_kwargs={}, response_metadata={}, id='87793c03-3824-40df-be11-a9c243a43b61'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 163, 'total_tokens': 318, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EL318uzjlBloxvSVAg2pj121yRdqi', 'service_tier': 'default', 'finish_reason': 'tool_call

## Headless Tools

Payment

Access the clipboard

Location

# Agents

# Dynamic Tool Gating

In [28]:
from langchain.tools import tool

@tool
def book_premium_concierge(destination: str) -> str:
    """Book a dedicated human concierge for trip planning. Premium members only."""
    return f"Premium concierge assigned for your {destination} trip."

In [29]:
from langchain.agents.middleware import (
    wrap_model_call,
    ModelRequest,
    ModelResponse,
)
from typing import Callable

In [30]:
@wrap_model_call
def gate_premium_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Only expose book_premium_concierge to premium members."""
    
    is_premium = request.state.get("is_premium_member", False)

    if not is_premium:
        allowed = [
            t for t in request.tools
            if t.name != "book_premium_concierge"
        ]
        request = request.override(tools=allowed)

    return handler(request)

In [31]:
from dataclasses import dataclass

@dataclass
class TravelerContext:
    user_id: str
    home_currency: str
    membership_tier: str
    is_premium_member: bool

In [ ]:
from pydantic import BaseModel, Field
from typing import Union, Literal
from langchain.agents.structured_output import ToolStrategy


class NewTripRequest(BaseModel):
    """A request to plan a new trip."""
    user_id: str
    destination: str
    start_date: str = Field(description="Format: YYYY-MM-DD")
    end_date: str = Field(description="Format: YYYY-MM-DD")


class ModifyTripRequest(BaseModel):
    """A request to change an existing trip."""
    user_id: str
    trip_id: int
    change_description: str


class CancelTripRequest(BaseModel):
    """A request to cancel an existing trip."""
    user_id: str
    trip_id: int

In [ ]:
full_tripmate = create_agent(
    model="openai:gpt-5-mini",
    tools=[
        get_real_weather,
        save_trip,
        get_saved_trips,
        save_travel_style,
        recall_travel_style,
        book_premium_concierge,
    ],
    system_prompt=(
        "You are TripMate, a real travel planning assistant. "
        "Check real weather before recommending destinations. "
        "Save trips when confirmed. "
        "Remember travel style preferences."
    ),
    middleware=[gate_premium_tools],
    checkpointer=InMemorySaver(),
    store=travel_store,
    context_schema=TravelerContext,
    name="tripmate_agent",
    response_format = ToolStrategy(
        Union[
            NewTripRequest,
            ModifyTripRequest,
            CancelTripRequest
        ]
    )
    
)